# 自定义中间件-Wrap-style hooks

## 1.wrap_model_call的使用

### 1.1基于装饰器的实现

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatDeepSeek(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    api_base = DEEPSEEK_BASE_URL,
    extra_body = {"thinking": {"type": "disabled"}}
)

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse, AgentMiddleware
from typing import Callable

@wrap_model_call
def warp_model_call_middleware(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:
    request.messages[-1].content += "----> warp_model_call_before <----"

    # 模型的调用
    res = handler(request)
    res.result[0].content += "---> wrap_model_call_after <---"

    return res

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model = model,
    middleware = [
        warp_model_call_middleware
    ]
)

res = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in res["messages"]:
    msg.pretty_print()

### 1.2基于类的实现

In [ ]:
from langchain.agents.middleware import AgentMiddleware

class WrapModelMiddleware(AgentMiddleware):
    def wrap_model_call(self,request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse | None:
        request.messages[-1].content += "----> warp_model_call_before <----"

        # 模型的调用
        res = handler(request)
        res.result[0].content += "---> wrap_model_call_after <---"

        return res

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model = model,
    middleware = [
        WrapModelMiddleware()
    ]
)

res = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in res["messages"]:
    msg.pretty_print()

## 2.wrap_tool_call的使用

### 2.1基于装饰器的实现

In [ ]:

from langchain_core.tools import tool
from typing import Any
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langgraph.prebuilt.tool_node import ToolCallRequest
from langchain.agents.middleware import wrap_tool_call


@tool
def get_weather(city: str, is_forcast: bool) -> str:
    """
    获取当日特定城市的天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明天的天气预报
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天天气也很好"
    return res


@wrap_tool_call
def wrap_tool_call_middleware(request: ToolCallRequest,
                              handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                              ) -> ToolMessage | Command[Any]:
    result = handler(request)
    print(f"原始参数：{request.tool_call['args']}")
    print(f"原始参数调用结果：{result}")

    request.tool_call["args"]["is_forcast"] = True
    result = handler(request)

    print(f"更新以后的参数：{request.tool_call['args']}")
    print(f"更新以后的参数调用结果：{result}")
    return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[wrap_tool_call_middleware]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询北京今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()

### 2.2 基于类的实现

In [ ]:
class WrapToolCallMiddleware(AgentMiddleware):
    def wrap_tool_call(self, request: ToolCallRequest,
                       handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
                       ) -> ToolMessage | Command[Any]:
        result = handler(request)
        print(f"原始参数：{request.tool_call['args']}")
        print(f"原始参数调用结果：{result}")

        request.tool_call["args"]["is_forcast"] = True
        result = handler(request)

        print(f"更新以后的参数：{request.tool_call['args']}")
        print(f"更新以后的参数调用结果：{result}")
        return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[WrapToolCallMiddleware()]
)

response = agent.invoke({
    "messages": [HumanMessage("帮我查询上海今天的天气如何？")]
})

for msg in response["messages"]:
    msg.pretty_print()